# Data Research — Exploratory Analysis

Bu defter **ince**: gerçek mantık `src/gift_contamination/` altındaki modüllerde.
Buradaki hücreler onları çağırır ve çıktıyı gösterir. Böylece analiz hem tek komutla
yeniden üretilebilir hem de burada gezilebilir kalıyor.

Teslim: `data-research/data-research.md`

```bash
python -m gift_contamination.data.download       --category all
python -m gift_contamination.data.preprocess     --category all
python -m gift_contamination.analysis.keyword_scan --category all
python -m gift_contamination.analysis.precision_check sample --category all   # elle etiketle
python -m gift_contamination.analysis.precision_check score
python -m gift_contamination.analysis.eda
python -m gift_contamination.analysis.deep_eda
```


In [ ]:
import polars as pl
from IPython.display import Image, Markdown, display

from gift_contamination.config import Config
from gift_contamination.analysis import deep_eda, eda, viz
from gift_contamination.utils.io import read_json

cfg = Config.load()
roles = eda.available_roles(cfg)
roles


## T1 — Ön işleme hunisi

Her filtrenin kaç satır, kullanıcı ve item elediği. Son satır (`05_k_core_5`)
kritik: pilot kategoride sıfıra iniyor.


In [ ]:
pl.DataFrame(eda.table_funnel(cfg, roles))


## T2 — Korpus profili

`users ≥5` sütunu 5-core'un neden bu kadar sert elediğini tek başına açıklıyor.


In [ ]:
pl.DataFrame(eda.table_corpus(cfg, roles))


## T3 — Anahtar kelime vekili oranları

`inflation` sütunu: tek bir "gift" regex'i koşsaydık oran kaç kat şişerdi.


In [ ]:
pl.DataFrame(eda.table_keyword(cfg, roles))


## T4 — Elle doğrulama (precision)

**Yazar destekli ön geçiş.** 3 annotator'lı insan doğrulaması Hafta 4'te.
Ekip aynı CSV'yi yeniden etiketleyip `precision_check score` ile tazeleyebilir.


In [ ]:
read_json(cfg.path('results', 'keyword_precision.json'))


---
## Derin analiz (T5-T14)

`eda.py` korpusun genel profilini verir. `deep_eda.py` ise projenin sonraki
asamalarinin dayandigi VARSAYIMLARI olcer. En kritik ucu:

- **T5** kanit metnin neresinde -> ModernBERT'in uzun-context gerekcesini sinar
- **T8** clean vs k-core -> 5-core hediye alicilarini sistematik eliyor mu
- **T11** ayni-gun kumelenmesi -> leave-one-out gercekten 'sonraki alim' mi


In [ ]:
for name, fn in deep_eda.TABLES:
    rows = fn(cfg, roles)
    if rows:
        display(Markdown(f'### {name}'), pl.DataFrame(rows))


### Derin analiz figurleri (F9-F16)


In [ ]:
for name, _ in deep_eda.FIGURES:
    path = cfg.path('figures', f"{name}.{cfg.get('eda.figure_format')}")
    if path.exists():
        display(Markdown(f'### {name}'), Image(filename=str(path)))


## Figürler

F6 ve F7 başlık figürleri: mevsimsellik (V2 ön izlemesi) ve kategori sıralaması (V3).


In [ ]:
for name, _ in eda.FIGURES:
    path = cfg.path('figures', f"{name}.{cfg.get('eda.figure_format')}")
    if path.exists():
        display(Markdown(f'### {name}'), Image(filename=str(path)))


## Figürleri yeniden üret

Config'te bir parametre değiştirdiysen (örn. `eda.min_year`) bu hücre yeter.


In [ ]:
eda.run(cfg)
deep_eda.run(cfg)
